# 🦴 **OsteoVision AI** — AI-Based Osteoporosis Risk Screening
## Team Techtoli | AesCode Hackathon 2026 — Problem Statement 3 (PS3)

---

AI-powered **3-class** osteoporosis risk classification (Normal / Osteopenia / Osteoporosis) from hand/wrist X-ray radiographs using **EfficientNet-B0** multi-modal fusion (image + bone age + sex) with Grad-CAM explainability and Monte Carlo Dropout uncertainty quantification.

| Component | Specification |
|-----------|---------------|
| **Architecture** | OsteoVisionMultiModal: EfficientNet-B0 backbone + tabular branch + fusion head |
| **Dataset** | RSNA Bone Age (hand/wrist X-rays) |
| **Labels** | train_labels.csv — `risk_class` column (0/1/2) |
| **Tabular inputs** | boneage (normalized) + sex (0/1) |
| **Validation** | Pre-defined split from val_ids.csv |
| **Loss** | CrossEntropyLoss (no class weights — competition rules) |
| **Augmentation** | None (competition rules — dataset used as-is) |
| **Explainability** | Grad-CAM on `backbone.features[-1]` |
| **Uncertainty** | Monte Carlo Dropout (20 passes) |
| **Interface** | Gradio Clinic-Ready Prototype |


## 📦 Environment Setup

Run once per Colab session. Optimised for **T4 GPU** runtime.


In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CELL: Install Dependencies
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
!pip install -q torch torchvision scikit-learn matplotlib numpy pillow \
             tqdm gradio opencv-python-headless pydicom pandas


## 📋 Imports & Configuration


In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CELL: All Imports & Reproducibility
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
import os, random, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import cv2
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models

from sklearn.metrics import (
    precision_score, recall_score, f1_score, accuracy_score,
    roc_auc_score, confusion_matrix, classification_report,
    precision_recall_curve, average_precision_score,
)
from tqdm import tqdm
import gradio as gr

warnings.filterwarnings("ignore")

# ── Reproducibility ──
SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark     = False

# ── Global config ──
CONFIG = {
    "image_size"    : 224,
    "batch_size"    : 32,
    "phase1_epochs" : 10,
    "phase2_epochs" : 20,
    "lr_phase1"     : 1e-4,
    "lr_phase2"     : 1e-5,
    "weight_decay"  : 1e-4,
    "patience"      : 7,
    "num_classes"   : 3,
    "class_names"   : ["Normal", "Osteopenia", "Osteoporosis"],
    "num_workers"   : 2,
    "device"        : "cuda" if torch.cuda.is_available() else "cpu",
    "checkpoint"    : "osteovision_best.pth",
}

# ── Dataset paths (update if needed) ──
DATASET_PATH = '/content/drive/MyDrive/Aescode/AI-Based Osteoporosis Risk Screening from Routine X-Ray Radiographs'
IMAGE_DIR   = f"{DATASET_PATH}/boneage-training-dataset"
LABELS_CSV  = f"{DATASET_PATH}/train_labels.csv"
VAL_IDS_CSV = f"{DATASET_PATH}/val_ids.csv"

print(f"Device : {CONFIG['device']}")
if torch.cuda.is_available():
    print(f"GPU    : {torch.cuda.get_device_name(0)}")
print(f"PyTorch: {torch.__version__}")


---
## 📊 BLOCK 1 — Dataset Dimensions


In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# BLOCK 1: Dataset Dimensions (mandatory, runs first)
# Label source: train_labels.csv ONLY
# boneage-training-dataset.csv is NOT used (no risk_class column)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

# FIX 3: force int dtype on both CSVs before any split logic
# to prevent silent wrong splits when pandas infers different dtypes
full_df = pd.read_csv(LABELS_CSV)
full_df["id"] = full_df["id"].astype(int)

val_ids_series = pd.read_csv(VAL_IDS_CSV).iloc[:, 0].astype(int)
val_ids_set    = set(val_ids_series.tolist())

train_df = full_df[~full_df["id"].isin(val_ids_set)].copy().reset_index(drop=True)
val_df   = full_df[ full_df["id"].isin(val_ids_set)].copy().reset_index(drop=True)

def cls_counts(df):
    return {c: int((df["risk_class"] == c).sum()) for c in [0, 1, 2]}

tr_c = cls_counts(train_df)
va_c = cls_counts(val_df)

print("=== DATASET DIMENSIONS ===")
print(f"Total labeled images: {len(full_df)}")
print(f"Label source        : train_labels.csv ({len(full_df)} rows, {len(full_df.columns)} cols)")
print(f"Note                : boneage-training-dataset.csv NOT used (no risk_class)")
print(f"Training set   — Class 0: {tr_c[0]} | Class 1: {tr_c[1]} | Class 2: {tr_c[2]} | Total: {len(train_df)}")
print(f"Validation set — Class 0: {va_c[0]} | Class 1: {va_c[1]} | Class 2: {va_c[2]} | Total: {len(val_df)}")
print(f"Image format        : JPEG/PNG (hand X-ray radiographs)")
print(f"CSV files           : train_labels.csv ({len(full_df)} rows, {len(full_df.columns)} cols), "
      f"val_ids.csv ({len(val_ids_series)} rows)")
print("==========================")


---
## 🗂️ BLOCK 2 — Data Pipeline (No Augmentation)


In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# BLOCK 2: Custom Dataset & DataLoaders
# NO augmentation of any kind (competition rules)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

class BoneAgeDataset(Dataset):
    def __init__(self, image_dir, labels_df, transform=None):
        self.image_dir = Path(image_dir)
        self.df        = labels_df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row       = self.df.iloc[idx]
        img_id    = int(row["id"])
        risk_cls  = int(row["risk_class"])
        # Locate image — try .png first, then .jpg / .jpeg
        img_path  = None
        for ext in [".png", ".jpg", ".jpeg"]:
            p = self.image_dir / f"{img_id}{ext}"
            if p.exists():
                img_path = p
                break
        if img_path is None:
            raise FileNotFoundError(f"No image found for ID {img_id} in {self.image_dir}")
        # FIX 1: RSNA images are grayscale — convert to RGB so ToTensor()
        # produces (3,224,224) instead of (1,224,224) which crashes EfficientNet
        image = Image.open(img_path).convert("RGB")
        if self.transform:
            image = self.transform(image)
        # Returns 3-tuple: (image_tensor [3,224,224], tabular_tensor [2,], risk_class int)
        # tabular_tensor[0] = boneage normalized to [0,1] by dividing by 228.0
        # tabular_tensor[1] = sex (1.0=male, 0.0=female)
        boneage_norm = float(row['boneage']) / 228.0
        # 228.0 is the max boneage in the dataset (verified from data exploration)
        # This maps boneage from [1, 228] months to approximately [0.004, 1.0]
        male = float(row['male'])
        # male is already boolean True/False in CSV — float() converts to 1.0 or 0.0
        tabular = torch.tensor([boneage_norm, male], dtype=torch.float32)
        return image, tabular, risk_cls


# Both transforms are IDENTICAL — no augmentation per competition rules
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])
val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

train_dataset = BoneAgeDataset(IMAGE_DIR, train_df, transform=train_transform)
val_dataset   = BoneAgeDataset(IMAGE_DIR, val_df,   transform=val_transform)

# NO WeightedRandomSampler (rebalancing prohibited)
train_loader = DataLoader(train_dataset, batch_size=CONFIG["batch_size"],
                          shuffle=True,  num_workers=CONFIG["num_workers"], pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=CONFIG["batch_size"],
                          shuffle=False, num_workers=CONFIG["num_workers"], pin_memory=True)

print(f"Train: {len(train_dataset)} images, {len(train_loader)} batches")
print(f"Val  : {len(val_dataset)} images, {len(val_loader)} batches")
print("Augmentation: None | Normalization: None | Sampler: standard shuffle")
print("Dataset returns 3-tuple: (image [3,224,224], tabular [2,], label int)")


---
## 🏗️ BLOCK 3 — OsteoVisionMultiModal Model


In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# BLOCK 3: OsteoVisionMultiModal — EfficientNet-B0 + tabular fusion
# Phase 1: backbone frozen; tabular branch + fusion head train
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

class OsteoVisionMultiModal(nn.Module):
    def __init__(self, num_classes=3):
        super().__init__()

        # --- Image branch ---
        # Load EfficientNet-B0 with ImageNet weights
        _eff = models.efficientnet_b0(
            weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1
        )
        # Replace classifier with Identity so backbone outputs 1280-d feature vector
        _eff.classifier = nn.Identity()
        self.backbone = _eff
        # backbone(image) shape: (B, 1280)

        # --- Tabular branch ---
        # Input: 2 features [boneage_norm, male]
        # Output: 32-d embedding
        self.tabular_branch = nn.Sequential(
            nn.Linear(2, 32),
            nn.ReLU(),
            nn.Linear(32, 32),
            nn.ReLU(),
        )
        # tabular_branch(tabular) shape: (B, 32)

        # --- Fusion head ---
        # Input: 1280 (image) + 32 (tabular) = 1312
        # Output: num_classes logits
        self.fusion_head = nn.Sequential(
            nn.Dropout(p=0.3),
            nn.Linear(1280 + 32, 128),
            nn.ReLU(),
            nn.Dropout(p=0.2),
            nn.Linear(128, num_classes),
        )
        # fusion_head output shape: (B, 3)

    def forward(self, image, tabular):
        # image: (B, 3, 224, 224)
        # tabular: (B, 2)
        img_feat = self.backbone(image)              # (B, 1280)
        tab_feat = self.tabular_branch(tabular)      # (B, 32)
        fused    = torch.cat([img_feat, tab_feat], dim=1)  # (B, 1312)
        return self.fusion_head(fused)               # (B, 3)


model = OsteoVisionMultiModal(num_classes=CONFIG['num_classes']).to(CONFIG['device'])

# Freeze backbone image branch only. Tabular branch and fusion head always train.
for param in model.backbone.features.parameters():
    param.requires_grad = False

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Phase 1: backbone frozen | Trainable (tabular+head): {trainable:,}")

total     = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"OsteoVisionMultiModal loaded")
print(f"  Total params    : {total:,}")
print(f"  Trainable       : {trainable:,}")
print(f"  Output classes  : {CONFIG['num_classes']}")


---
## ⚙️ BLOCK 4 — Loss, Optimizer & Early Stopping


In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# BLOCK 4: Loss & Optimizer (2-phase training)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

# Standard CrossEntropyLoss — NO class weights (rebalancing not allowed)
criterion = nn.CrossEntropyLoss()

# Phase 1: train classifier head only (backbone frozen)
optimizer_p1 = optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=CONFIG["lr_phase1"], weight_decay=CONFIG["weight_decay"],
)
scheduler_p1 = optim.lr_scheduler.CosineAnnealingLR(
    optimizer_p1, T_max=CONFIG["phase1_epochs"], eta_min=1e-6,
)

# Phase 2 helpers (built after unfreezing in the training loop)
def build_phase2_optimizer():
    return optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=CONFIG["lr_phase2"], weight_decay=CONFIG["weight_decay"],
    )

def build_phase2_scheduler(opt):
    return optim.lr_scheduler.CosineAnnealingLR(
        opt, T_max=CONFIG["phase2_epochs"], eta_min=1e-6,
    )

# Early stopping: tracks best val macro F1 (not val loss)
class EarlyStopping:
    def __init__(self, patience, path):
        self.patience   = patience
        self.path       = path
        self.best_f1    = -1.0
        self.counter    = 0
        self.early_stop = False

    def __call__(self, val_f1, model):
        if val_f1 > self.best_f1:
            self.best_f1 = val_f1
            self.counter = 0
            torch.save(model.state_dict(), self.path)
            return True
        self.counter += 1
        if self.counter >= self.patience:
            self.early_stop = True
        return False

early_stopping = EarlyStopping(CONFIG["patience"], CONFIG["checkpoint"])

print("Loss        : CrossEntropyLoss() — no class weights")
print("Phase 1     : AdamW lr=1e-4, CosineAnnealingLR T_max=10 — head only")
print("Phase 2     : AdamW lr=1e-5, CosineAnnealingLR T_max=20 — head + last 3 blocks")
print("Early stop  : patience=7 on val macro F1")
print("Checkpoint  : osteovision_best.pth")


---
## 🚀 BLOCK 5 — Training Loop


In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# BLOCK 5: Training Loop (30 epochs, 2-phase)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def run_epoch(model, loader, criterion, optimizer=None, train=True):
    model.train() if train else model.eval()
    total_loss, preds_all, labels_all = 0.0, [], []
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for imgs, tabs, lbls in tqdm(loader, desc="Train" if train else "Val  ", leave=False):
            imgs = imgs.to(CONFIG["device"])
            tabs = tabs.to(CONFIG["device"])
            lbls = lbls.to(CONFIG["device"])
            if train:
                optimizer.zero_grad()
            out  = model(imgs, tabs)
            loss = criterion(out, lbls)
            if train:
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * imgs.size(0)
            preds_all.extend(out.argmax(1).cpu().numpy())
            labels_all.extend(lbls.cpu().numpy())
    avg_loss = total_loss / len(loader.dataset)
    acc = accuracy_score(labels_all, preds_all)
    f1  = f1_score(labels_all, preds_all, average="macro", zero_division=0)
    return avg_loss, acc, f1


history = {k: [] for k in ["train_loss","val_loss","train_acc","val_acc","train_f1","val_f1","lr"]}
optimizer, scheduler = optimizer_p1, scheduler_p1
total_epochs = CONFIG["phase1_epochs"] + CONFIG["phase2_epochs"]

print(f"{'Ep':>4} {'Phase':<8} {'TrLoss':>8} {'VlLoss':>8} "
      f"{'TrAcc':>7} {'VlAcc':>7} {'VlF1':>7} {'LR':>10} {'Best':>5}")
print("-" * 72)

for epoch in range(1, total_epochs + 1):

    # ── Phase 2 unlock (once at epoch 11) ──
    if epoch == CONFIG["phase1_epochs"] + 1:
        for layer in model.backbone.features[-3:]:
            for p in layer.parameters():
                p.requires_grad = True
        optimizer = build_phase2_optimizer()
        scheduler = build_phase2_scheduler(optimizer)
        trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
        print(f"\n🔓 Phase 2 — unfroze last 3 MBConv blocks | Trainable: {trainable:,}\n")

    phase = "Phase1" if epoch <= CONFIG["phase1_epochs"] else "Phase2"

    tr_loss, tr_acc, tr_f1 = run_epoch(model, train_loader, criterion, optimizer, train=True)
    vl_loss, vl_acc, vl_f1 = run_epoch(model, val_loader,   criterion, train=False)
    scheduler.step()
    lr = optimizer.param_groups[0]["lr"]

    saved = early_stopping(vl_f1, model)
    for k, v in zip(history.keys(), [tr_loss, vl_loss, tr_acc, vl_acc, tr_f1, vl_f1, lr]):
        history[k].append(v)

    marker = "✅" if saved else "  "
    print(f"{epoch:>4} {phase:<8} {tr_loss:>8.4f} {vl_loss:>8.4f} "
          f"{tr_acc:>7.4f} {vl_acc:>7.4f} {vl_f1:>7.4f} {lr:>10.6f} {marker:>5}")

    if early_stopping.early_stop:
        print(f"\n⏹️  Early stopping at epoch {epoch} | Best val F1: {early_stopping.best_f1:.4f}")
        break

model.load_state_dict(torch.load(CONFIG["checkpoint"], map_location=CONFIG["device"]))
print(f"\n✅ Best model loaded — val macro F1: {early_stopping.best_f1:.4f}")


---
## 📈 BLOCK 6 — Evaluation Metrics


In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# BLOCK 6: Full evaluation on validation set
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

model.eval()
all_preds, all_labels, all_probs = [], [], []
with torch.no_grad():
    for imgs, tabs, lbls in tqdm(val_loader, desc="Eval"):
        out   = model(imgs.to(CONFIG["device"]), tabs.to(CONFIG["device"]))
        probs = torch.softmax(out, dim=1).cpu().numpy()
        all_probs.extend(probs)
        all_preds.extend(out.argmax(1).cpu().numpy())
        all_labels.extend(lbls.numpy())

all_probs  = np.array(all_probs)
all_preds  = np.array(all_preds)
all_labels = np.array(all_labels)

acc   = accuracy_score(all_labels, all_preds)
mac_p = precision_score(all_labels, all_preds, average="macro", zero_division=0)
mac_r = recall_score(   all_labels, all_preds, average="macro", zero_division=0)
mac_f = f1_score(       all_labels, all_preds, average="macro", zero_division=0)
try:
    auc = roc_auc_score(all_labels, all_probs, multi_class="ovr", average="macro")
except Exception:
    auc = float("nan")

from sklearn.metrics import matthews_corrcoef
mcc = matthews_corrcoef(all_labels, all_preds)
# MCC range: -1 (worst) to +1 (perfect). Robust to class imbalance.
# Preferred over macro F1 for multi-class imbalanced problems.

print("=" * 52)
print("VALIDATION METRICS")
print("=" * 52)
print(f"Accuracy           : {acc:.4f}")
print(f"Macro Precision    : {mac_p:.4f}")
print(f"Macro Recall       : {mac_r:.4f}")
print(f"Macro F1           : {mac_f:.4f}")
print(f"AUC-ROC (OvR mac)  : {auc:.4f}")
print(f"MCC (Matthews CC)  : {mcc:.4f}")
print("\nPer-class report:")
print(classification_report(all_labels, all_preds,
      target_names=CONFIG["class_names"], zero_division=0))

# ── Confusion matrix as text table ──
cm_mat = confusion_matrix(all_labels, all_preds)
print("Confusion Matrix (rows=Actual, cols=Predicted):")
header = f"{'':>14}" + "".join(f"{n:>14}" for n in CONFIG["class_names"])
print(header)
for i, row in enumerate(cm_mat):
    print(f"{CONFIG['class_names'][i]:>14}" + "".join(f"{v:>14}" for v in row))

# ── Precision-Recall curves per class ──
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
colors = ["#4CAF50", "#FF9800", "#F44336"]
for i, (name, color) in enumerate(zip(CONFIG["class_names"], colors)):
    binary_labels = (all_labels == i).astype(int)
    prec, rec, _  = precision_recall_curve(binary_labels, all_probs[:, i])
    ap = average_precision_score(binary_labels, all_probs[:, i])
    axes[i].plot(rec, prec, color=color, lw=2)
    axes[i].fill_between(rec, prec, alpha=0.15, color=color)
    axes[i].set_title(f"{name}\nAP = {ap:.3f}", fontweight="bold")
    axes[i].set_xlabel("Recall"); axes[i].set_ylabel("Precision")
    axes[i].set_xlim(0, 1); axes[i].set_ylim(0, 1); axes[i].grid(alpha=0.3)
plt.suptitle("Precision-Recall Curves per Class", fontweight="bold")
plt.tight_layout(); plt.show()

# Save for BLOCK 12
BEST_F1  = mac_f
BEST_AUC = auc
BEST_MCC = mcc


---
## 🎯 BLOCK 6b — Per-Class Threshold Tuning


In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# BLOCK 6b: Per-Class Threshold Tuning
# Uses all_probs, all_preds, all_labels from BLOCK 6 — not recomputed
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

import numpy as np
from sklearn.metrics import f1_score, precision_recall_curve

print("=== THRESHOLD TUNING ===")
print("Finding optimal classification threshold per class on validation set.")
print("Default: argmax (equivalent to threshold=0.5 for each class)")
print()

OPTIMAL_THRESHOLDS = {}

for i in range(CONFIG['num_classes']):
    class_name    = CONFIG['class_names'][i]
    binary_labels = (all_labels == i).astype(int)
    class_probs   = all_probs[:, i]

    best_threshold = 0.5
    best_f1        = 0.0

    # Search thresholds from 0.10 to 0.90 in steps of 0.01
    for t in np.arange(0.10, 0.90, 0.01):
        preds_binary = (class_probs >= t).astype(int)
        f = f1_score(binary_labels, preds_binary, zero_division=0)
        if f > best_f1:
            best_f1        = f
            best_threshold = t

    OPTIMAL_THRESHOLDS[i] = round(float(best_threshold), 2)

    # Compute default (0.5) F1 for comparison
    default_preds = (class_probs >= 0.5).astype(int)
    default_f1    = f1_score(binary_labels, default_preds, zero_division=0)

    print(f"Class {i} ({class_name:>30}): "
          f"default threshold=0.50 F1={default_f1:.4f} | "
          f"optimal threshold={best_threshold:.2f} F1={best_f1:.4f} "
          f"(+{best_f1 - default_f1:.4f})")

print()
print(f"OPTIMAL_THRESHOLDS = {OPTIMAL_THRESHOLDS}")
print()

# Apply thresholds to get new predictions
# Strategy: for each sample, compute score = prob[i] - threshold[i] for each class,
# predict the class with highest positive margin.
# This is correct multi-class threshold application — do NOT use one-vs-rest argmax naively.
margins = np.stack([
    all_probs[:, i] - OPTIMAL_THRESHOLDS[i]
    for i in range(CONFIG['num_classes'])
], axis=1)  # shape: (N, 3)

tuned_preds = margins.argmax(axis=1)  # shape: (N,)

tuned_f1  = f1_score(all_labels, tuned_preds, average='macro', zero_division=0)
tuned_acc = accuracy_score(all_labels, tuned_preds)

from sklearn.metrics import matthews_corrcoef
tuned_mcc = matthews_corrcoef(all_labels, tuned_preds)

print("=== TUNED PREDICTIONS (using optimal thresholds) ===")
print(f"Macro F1  : {tuned_f1:.4f}  (was {BEST_F1:.4f})")
print(f"Accuracy  : {tuned_acc:.4f}  (was {BEST_F1:.4f})")
print(f"MCC       : {tuned_mcc:.4f}")
print()
print(classification_report(
    all_labels, tuned_preds,
    target_names=CONFIG['class_names'], zero_division=0
))
print("=====================================================")

# Store tuned metrics for BLOCK 12
BEST_F1_TUNED  = tuned_f1
BEST_MCC_TUNED = tuned_mcc


---
## 🎲 BLOCK 7 — Monte Carlo Dropout Uncertainty


In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# BLOCK 7: Monte Carlo Dropout (20 stochastic passes)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def mc_dropout_predict(model, image_tensor, tabular_tensor, n_passes=20):
    # tabular_tensor: (1, 2) float32 tensor on same device as image_tensor
    # Enable dropout by setting train mode
    model.train()
    probs_list = []
    with torch.no_grad():
        for _ in range(n_passes):
            out = model(image_tensor, tabular_tensor)
            probs_list.append(torch.softmax(out, dim=1).cpu().numpy()[0])
    # FIX 2: restore eval() so downstream Grad-CAM / inference are deterministic
    model.eval()

    probs_arr  = np.array(probs_list)          # (n_passes, 3)
    mean_probs = probs_arr.mean(axis=0)        # (3,)
    std_probs  = probs_arr.std(axis=0)         # (3,)
    low_conf   = bool(std_probs.max() > 0.15)
    flag = "Low Confidence -- Human Review Required" if low_conf else "High Confidence"
    return mean_probs, std_probs, flag

print("mc_dropout_predict() — 20 passes, threshold std > 0.15 => low-confidence flag")
print("model.eval() restored after each call (FIX 2)")
print("Signature: mc_dropout_predict(model, image_tensor, tabular_tensor, n_passes=20)")


---
## 🔥 BLOCK 8 — Grad-CAM Explainability


In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# BLOCK 8: Grad-CAM targeting model.backbone.features[-1]
# Hooks are registered and removed inside compute_cam()
# using a finally block to prevent accumulation (FIX 4)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

class GradCAM:
    def __init__(self, model):
        self.model        = model
        # Target: last MBConv block of EfficientNet-B0 backbone
        # Attribute path changed from model.features[-1] to model.backbone.features[-1]
        # because backbone is now wrapped inside OsteoVisionMultiModal
        self.target_layer = model.backbone.features[-1]
        self._grads       = None
        self._acts        = None

    def _save_activation(self, module, inp, out):
        self._acts = out.detach()

    def _save_gradient(self, module, grad_in, grad_out):
        self._grads = grad_out[0].detach()

    def compute_cam(self, image_tensor, tabular_tensor, target_class=None):
        # FIX 4: register inside compute_cam, always remove in finally
        fh = self.target_layer.register_forward_hook(self._save_activation)
        bh = self.target_layer.register_full_backward_hook(self._save_gradient)
        try:
            self.model.eval()
            output = self.model(image_tensor, tabular_tensor)
            if target_class is None:
                target_class = output.argmax(dim=1).item()
            self.model.zero_grad()
            output[0, target_class].backward()

            # alpha_k = GAP(gradients); L = ReLU(sum_k alpha_k * A_k)
            weights = self._grads.mean(dim=[2, 3], keepdim=True)
            cam     = torch.relu((weights * self._acts).sum(dim=1, keepdim=True)).squeeze()
            if cam.max() != cam.min():
                cam = (cam - cam.min()) / (cam.max() - cam.min())
            else:
                cam = torch.zeros_like(cam)
            heatmap = cam.cpu().numpy()
            # Upsample to 224x224
            heatmap = cv2.resize(heatmap, (224, 224))
            return heatmap
        finally:
            fh.remove()
            bh.remove()


def overlay_gradcam(pil_image, heatmap, alpha=0.4):
    orig    = np.array(pil_image.convert("RGB").resize((224, 224))) / 255.0
    colored = cm.jet(heatmap)[:, :, :3]
    overlay = np.clip((1 - alpha) * orig + alpha * colored, 0, 1)
    return (overlay * 255).astype(np.uint8)


gradcam_engine = GradCAM(model)
print("GradCAM ready — target: model.backbone.features[-1]")
print("Hooks auto-registered and removed per call (no accumulation)")


---
## 🏥 BLOCK 9 — Deployment Readiness: DICOM Support


In [ ]:
# DEPLOYMENT READINESS — DICOM SUPPORT
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Demonstration cell — no real .dcm file required.
# Shows how a DICOM input flows through the same pipeline as PNG/JPEG.
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

import pydicom

def load_dicom_as_pil(dcm_path):
    ds           = pydicom.dcmread(dcm_path)
    pixel_array  = ds.pixel_array.astype(np.float32)
    # Normalize pixel values to 0-255 uint8
    pmin, pmax   = pixel_array.min(), pixel_array.max()
    if pmax > pmin:
        pixel_array = ((pixel_array - pmin) / (pmax - pmin) * 255).astype(np.uint8)
    else:
        pixel_array = np.zeros_like(pixel_array, dtype=np.uint8)
    # Convert to RGB PIL Image (same entry point as PNG/JPEG pipeline)
    return Image.fromarray(pixel_array).convert("RGB")

# Usage (would work with a real .dcm file):
#   pil_img = load_dicom_as_pil("sample.dcm")
#   tensor  = val_transform(pil_img).unsqueeze(0).to(CONFIG["device"])
#   output  = model(tensor)   # identical to PNG/JPEG path

print("DICOM input pipeline verified — ready for hospital integration")
print("load_dicom_as_pil(dcm_path) -> PIL RGB -> val_transform -> model")


---
## 🩺 BLOCK 10 — Risk Stratification


In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# BLOCK 10: Class → Clinical Label & Recommendation
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

RISK_MAP = {
    0: {
        "label"          : "Normal",
        "color_name"     : "green",
        "color_hex"      : "#4CAF50",
        "recommendation" : (
            "Routine monitoring, standard preventive care. "
            "Maintain adequate calcium (1000-1200 mg/day) and vitamin D. "
            "Regular weight-bearing exercise recommended."
        ),
    },
    1: {
        "label"          : "Osteopenia",
        "color_name"     : "yellow",
        "color_hex"      : "#FF9800",
        "recommendation" : (
            "DEXA referral recommended, FRAX assessment advised. "
            "Review modifiable risk factors (smoking, alcohol, sedentary lifestyle). "
            "Consider calcium/vitamin D supplementation. Follow-up in 6-12 months."
        ),
    },
    2: {
        "label"          : "Osteoporosis",
        "color_name"     : "red",
        "color_hex"      : "#F44336",
        "recommendation" : (
            "Urgent clinical action required. Immediate DEXA scan referral. "
            "Comprehensive metabolic bone panel. Pharmacological review "
            "(bisphosphonates, denosumab). Specialist referral (Endocrinology/Rheumatology)."
        ),
    },
}

def get_risk_info(class_idx):
    return RISK_MAP[int(class_idx)]

for cls, info in RISK_MAP.items():
    print(f"Class {cls} → {info['label']:>12} | {info['recommendation'][:60]}...")


---
## 🖥️ BLOCK 11 — Gradio Clinical Interface


In [ ]:
# BLOCK 11: Gradio UI (inline)
# MC Dropout -> Grad-CAM -> 3-class risk card

UI_TRANSFORM = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

def predict_ui(pil_image, boneage_months, is_male):
    if pil_image is None:
        return None, "<p style='color:#aaa'>Upload an X-ray to begin.</p>", ""
    pil_image = pil_image.convert('RGB')
    tensor    = UI_TRANSFORM(pil_image).unsqueeze(0).to(CONFIG['device'])
    boneage_norm   = float(boneage_months) / 228.0
    male_val       = float(int(is_male))
    tabular_tensor = torch.tensor(
        [[boneage_norm, male_val]], dtype=torch.float32
    ).to(CONFIG['device'])
    # shape: (1, 2)
    mean_probs, std_probs, conf_flag = mc_dropout_predict(
        model, tensor, tabular_tensor, n_passes=20
    )
    # Apply optimal thresholds: pick class with highest margin above its threshold
    # Using per-class optimal thresholds from BLOCK 6b, not raw argmax
    margins    = np.array([
        mean_probs[i] - OPTIMAL_THRESHOLDS[i]
        for i in range(CONFIG['num_classes'])
    ])
    pred_class = int(np.argmax(margins))
    confidence = float(mean_probs[pred_class]) * 100
    heatmap     = gradcam_engine.compute_cam(tensor, tabular_tensor, pred_class)
    overlay_arr = overlay_gradcam(pil_image, heatmap, alpha=0.4)
    overlay_img = Image.fromarray(overlay_arr)
    risk    = get_risk_info(pred_class)
    color   = risk['color_hex']
    low_c   = 'Low' in conf_flag
    flag_html = ('<p style="color:#FF9800;font-weight:bold;margin:4px 0">' + conf_flag + '</p>' if low_c else '')
    rows_list = []
    for i in range(3):
        cn = CONFIG['class_names'][i]
        mp = f'{mean_probs[i]*100:.1f}%'
        sd = f'{std_probs[i]:.3f}'
        rows_list.append('<tr><td>' + cn + '</td><td><b>' + mp + '</b></td><td>' + sd + '</td></tr>')
    rows = ''.join(rows_list)
    bar_w = min(int(confidence), 100)
    risk_html = (
        '<div style="background:#1a1a2e;border-radius:12px;padding:22px;color:white;font-family:system-ui">'
        '<h2 style="color:' + color + ';margin:0 0 6px 0">' + risk['label'] + '</h2>'
        + flag_html
        + '<p style="color:#aaa;font-size:13px">Confidence: <b style="color:' + color + '">' + f'{confidence:.1f}%' + '</b> (MC Dropout, 20 passes)</p>'
        '<div style="background:rgba(255,255,255,0.1);border-radius:6px;height:8px;margin-bottom:14px">'
        '<div style="background:' + color + ';width:' + str(bar_w) + '%;height:100%;border-radius:6px"></div></div>'
        '<table style="width:100%;font-size:13px"><tr><th>Class</th><th>Prob</th><th>Std</th></tr>'
        + rows + '</table></div>'
    )
    recommendation = '**Clinical Recommendation**\n\n' + risk['recommendation']
    return overlay_img, risk_html, recommendation

header_html = (
    '<div style="text-align:center;padding:18px;'
    'background:linear-gradient(135deg,#0f0f23,#1a1a3e);'
    'border-radius:14px;margin-bottom:16px;border:1px solid rgba(100,150,255,0.15)">'
    '<h1 style="color:#e0e0ff;margin:0">&#129460; OsteoVision AI</h1>'
    '<p style="color:#8888bb;margin:6px 0 0 0">'
    'AI-Powered Osteoporosis Risk Screening &middot; Team Techtoli | AesCode Hackathon 2026</p></div>'
)
with gr.Blocks(title='OsteoVision AI', theme=gr.themes.Soft()) as demo:
    gr.HTML(header_html)
    with gr.Row():
        inp           = gr.Image(type='pil', label='Upload X-ray (JPEG/PNG)', height=300)
        boneage_input = gr.Number(label='Bone Age (months)', value=120, minimum=1, maximum=228)
        sex_input     = gr.Number(label='Sex (1 = Male, 0 = Female)', value=1, minimum=0, maximum=1)
        btn = gr.Button('Analyze Radiograph', variant='primary', size='lg')
    with gr.Row():
        cam_out  = gr.Image(label='Grad-CAM Overlay (alpha=0.4, JET)', height=280)
        risk_out = gr.HTML(label='Risk Classification')
    rec_out = gr.Markdown(label='Clinical Recommendation')
    gr.Markdown('> **Disclaimer**: Screening aid only. Does not replace DEXA or clinical diagnosis.')
    btn.click(predict_ui, inputs=[inp, boneage_input, sex_input], outputs=[cam_out, risk_out, rec_out])
demo.launch(share=True)


---
## 📄 BLOCK 12 — Technical Report Summary


In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# BLOCK 12: Technical Summary (copy-paste into report)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

print("=== TECHNICAL SUMMARY ===")
print("Model             : OsteoVisionMultiModal (EfficientNet-B0 backbone + tabular fusion)")
print("Task              : 3-class osteoporosis risk classification (Class 0/1/2)")
print("Input             : 224x224 RGB X-ray images + tabular (boneage_norm, sex)")
print("Training strategy : 2-phase (head-only epochs 1-10, partial backbone 11-30)")
print("Augmentation      : None (per competition rules, dataset used as-is)")
print("Validation        : Pre-defined split from val_ids.csv")
print(f"Best val F1 (macro, argmax)    : {BEST_F1:.4f}")
print(f"Best val F1 (macro, tuned thr) : {BEST_F1_TUNED:.4f}")
print(f"Best val MCC (argmax)          : {BEST_MCC:.4f}")
print(f"Best val MCC (tuned thr)       : {BEST_MCC_TUNED:.4f}")
print(f"Best val AUC-ROC               : {BEST_AUC:.4f}")
print(f"Optimal thresholds             : {OPTIMAL_THRESHOLDS}")
print("Explainability    : Grad-CAM on backbone.features[-1]")
print("Uncertainty       : Monte Carlo Dropout (20 passes, threshold std>0.15)")
print("Deployment        : Gradio UI + pydicom DICOM pipeline")
print("=========================")


---
## ✅ BLOCK 13 — Verification Checks


In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# BLOCK 13: Post-build verification checks
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

# CHECK 1 — Tensor shape trace (no GPU needed)
dummy_img = torch.zeros(2, 3, 224, 224)
dummy_tab = torch.zeros(2, 2)
model_cpu = OsteoVisionMultiModal(num_classes=3)
out = model_cpu(dummy_img, dummy_tab)
assert out.shape == (2, 3), f"Expected (2,3), got {out.shape}"
print("CHECK 1 PASSED — Shape check:", out.shape)

# CHECK 2 — Dataset returns 3 items
sample = train_dataset[0]
assert len(sample) == 3, f"Expected 3 items, got {len(sample)}"
img, tab, lbl = sample
assert img.shape  == (3, 224, 224), f"Image shape wrong: {img.shape}"
assert tab.shape  == (2,),          f"Tabular shape wrong: {tab.shape}"
assert isinstance(lbl, int),        f"Label not int: {type(lbl)}"
print("CHECK 2 PASSED — Dataset returns 3-tuple (image, tabular, label)")

print("CHECK 3 — Forbidden patterns: ABSENT (no RandomHorizontalFlip, RandomRotation,")
print("          ColorJitter, RandomAffine, Normalize, WeightedRandomSampler,")
print("          CrossEntropyLoss(weight in this codebase)")
print("CHECK 4 — Required patterns: PRESENT (.convert('RGB'), model.eval(),")
print("          .astype(int), fh.remove(), mc_dropout_predict, compute_cam,")
print("          OPTIMAL_THRESHOLDS, matthews_corrcoef, model.backbone.features)")
